In [99]:
# add the root to the path
import sys
import os
import pandas as pd
import json
sys.path.append('/Users/pmassaro/Repos/JobSeeker')
from sqlalchemy import text
from jobseeker.scraper.database.database_manager import DatabaseManager
from jobseeker.scraper.database.models import JobPosting,Users
from jobseeker.llm import ModelNames
from jobseeker.llm.cv_data_extractor import CVLLMExtractor
from jobseeker.llm.job_description_extractor import JobDescriptionLLMExtractor
from jobseeker.llm.cv_builder import CVBuilder
from jobseeker.llm.requirements_qualification_comparator import RequirementsQualificationsComparator
from jobseeker.llm.cover_letter_builder import CoverLetterBuilder
from jobseeker.llm.utils import extract_text_from_pdf


db = DatabaseManager()
user_id = 1

In [103]:
# create a query selecting all the job postings
query = text("""
    SELECT 
        *
    FROM job_postings 
    where 
        lower(company) like '%anomalo%' 
        and lower(title) like '%machine%' 
        -- and id in (391)

""" )
session = db.get_session()
job_postings= session.execute(query)
job_postings = pd.DataFrame(job_postings.fetchall(), columns=job_postings.keys())
session.close()
job_postings

,id,job_id,title,seniority_level,employment_type,job_description,company,company_url,industries,job_functions,institution_id,job_salary_range_max,job_salary_range_min,job_poster_profile_url,job_poster_name,skills,date_created,job_posting_summary,date_updated
0,416,3905297605,Applied Machine Learning Engineer (East Coast),Entry level,Full-time,About Anomalo\n\n\nData management solutions a...,Anomalo,https://www.linkedin.com/company/anomalo,[Software Development],[Engineering Information Technology],None,250000,210000,None,None,None,2024-04-20 19:28:20.649206,None,2024-04-20 19:28:20.649206


# Create job description and CV summaries using LLMs

In [104]:
# job_posting_parser = JobDescriptionLLMExtractor(model_name=ModelNames.GPT4_TURBO,temperature=0)
# job_posting_parser.update_job_postings(job_postings)

# cv_extractor = CVLLMExtractor(model_name=ModelNames.GPT3_TURBO,temperature=0)
# cv_extractor.extract_cv_and_write_to_db(user_id=user_id)


2024-04-20 19:29:40,734 - JobDescriptionExtractor - INFO - Extracted data from text: 
 Tokens Used: 3375
	Prompt Tokens: 2828
	Completion Tokens: 547
Successful Requests: 1
Total Cost (USD): $0.0


Now, query the DB and get the relevant info

In [105]:
job_id = 3905297605
columns_to_select = [
    getattr(JobPosting, attr) for attr in JobPosting.__table__.columns.keys()
    if attr not in ['_sa_instance_state', 'company_url',"job_id","date_created","job_description","id","institution_id","date_updated"]  # List excluded columns
]
session = db.get_session()
data=session.query(*columns_to_select).filter_by(job_id=int(job_id)).first()
session.close()
job_posting_metadata= json.dumps(data._asdict())

columns_to_select = [
    getattr(Users, attr) for attr in Users.__table__.columns.keys()
    if attr in ["cv_summary"]  # List excluded columns
]

session = db.get_session()
data=session.query(*columns_to_select).filter_by(id=int(user_id)).first()
session.close()
cv_metadata = json.dumps(data._asdict())



#### COMPARISON
comparator = RequirementsQualificationsComparator(model_name=ModelNames.GPT4_TURBO,
                                                  cv_data=cv_metadata,
                                                  temperature=0.5)
req_qual_comparison = comparator.compare(text=job_posting_metadata)
        
# convert the comparison to a string
req_qual_comparison = json.dumps(req_qual_comparison['requirements_qualification_comparisons'])


2024-04-20 19:30:38,216 - RequirementsQualificationsComparator - INFO - Extracted data from text: 
 Tokens Used: 2737
	Prompt Tokens: 2216
	Completion Tokens: 521
Successful Requests: 1
Total Cost (USD): $0.0


In [106]:
with open("../media/1/CV.tex","r") as f:
    cv_latex = f.read()

In [109]:
import importlib
importlib.reload(sys.modules['jobseeker.llm.cv_builder'])
importlib.reload(sys.modules['jobseeker.llm.base_builder'])
from jobseeker.llm.cv_builder import CVBuilder
#### CV BUILDER #####
# get the 
cv_builder = CVBuilder(model_name=ModelNames.GPT4_TURBO,
                       temperature=0.8,
                       user_id=user_id,
                       user_cv_data=cv_metadata,
                       user_cv_latex=cv_latex)

cv_builder.build(job_data=job_posting_metadata,
                 requirement_qualification_comparison=req_qual_comparison,
                 job_id=job_id)
                       

2024-04-20 19:34:21,534 - CoverLetterBuilder - INFO - Job 3905297605
2024-04-20 19:34:57,953 - CoverLetterBuilder - INFO - Built CV for user 1 
 Tokens Used: 3719
	Prompt Tokens: 2804
	Completion Tokens: 915
Successful Requests: 1
Total Cost (USD): $0.0
2024-04-20 19:34:58,805 - CoverLetterBuilder - INFO - PDF created successfully: /Users/pmassaro/Repos/JobSeeker/jobseeker/media/1/3905297605_CV.pdf


: 

In [108]:


##### COVER LETTER BUILD ####
cover_letter_builder = CoverLetterBuilder(model_name=ModelNames.GPT4_TURBO,
                                          temperature=0.75,
                                          user_cv_data=cv_metadata,
                                          user_id=user_id,
                                          )
        
cover_letter_builder.build_letter(requirement_qualification_comparison=req_qual_comparison,
                                  job_data=job_posting_metadata,
                                  job_id=job_id
)

2024-04-20 19:31:59,302 - CoverLetterBuilder - INFO - Building letter with the following data: 
 {"title": "Applied Machine Learning Engineer (East Coast)", "seniority_level": "Entry level", "employment_type": "Full-time", "company": "Anomalo", "industries": ["Software Development"], "job_functions": ["Engineering Information Technology"], "job_salary_range_max": 250000, "job_salary_range_min": 210000, "job_poster_profile_url": null, "job_poster_name": null, "skills": null, "job_posting_summary": {"job_description": {"title": "Applied Machine Learning Engineer", "location": "Remote", "remote": true, "pay_range": "$210,000 - $250,000"}, "responsibilities": {"items": ["Solve complex problems using Machine Learning to address data quality issues.", "Lead the development of features enhancing operational efficiency.", "Enhance ML accessibility and democratize data quality monitoring.", "Own and enhance a suite of interpretable ML tools.", "Develop visualizations for complex ML findings.", 